In [1]:
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os
import torch.nn.functional as F

In [2]:
path = "/project/CoSiR/res/CoSiR_Experiment/coco/20260225_091925_CoSiR_Experiment/embeddings"
# Get file names in the path
file_names = os.listdir(path)
# Load the embeddings
embeddings = []
for file_name in file_names:
    with open(os.path.join(path, file_name), "rb") as f:
        embeddings.append(pickle.load(f))

In [3]:
predicted_conditions = embeddings[0]
features = embeddings[1]

In [4]:
predicted_conditions.keys()

dict_keys(['img_condition_predicted', 'txt_condition_predicted', 'imgtxt_condition_predicted', 'label_embeddings'])

In [ ]:
img_condition_predicted = predicted_conditions["img_condition_predicted"]
txt_condition_predicted = predicted_conditions["txt_condition_predicted"]
imgtxt_condition_predicted = predicted_conditions["imgtxt_condition_predicted"]
label_embeddings = predicted_conditions["label_embeddings"]
# Turn everything to tensor
img_condition_predicted = torch.from_numpy(img_condition_predicted)
txt_condition_predicted = torch.from_numpy(txt_condition_predicted)
imgtxt_condition_predicted = torch.from_numpy(imgtxt_condition_predicted)

In [19]:
# Compute one-to-one difference between all three predicted conditions and label embeddings, by one-to-one difference, I mean the i-th element of the predicted condition and the i-th element of the label embeddings
diff_img = np.linalg.norm(img_condition_predicted - label_embeddings, axis=1).mean()
diff_txt = np.linalg.norm(txt_condition_predicted - label_embeddings, axis=1).mean()
diff_imgtxt = np.linalg.norm(
    imgtxt_condition_predicted - label_embeddings, axis=1
).mean()

print(diff_img, diff_txt, diff_imgtxt)

TypeError: unsupported operand type(s) for -: 'numpy.ndarray' and 'Tensor'

In [16]:
features.keys()

dict_keys(['img_features', 'txt_features', 'combined_features_origin', 'combined_condition_img', 'combined_condition_txt', 'combined_condition_imgtxt'])

In [20]:
img_features = features["img_features"]
txt_features = features["txt_features"]
combined_features_origin = features["combined_features_origin"]
combined_condition_img = features["combined_condition_img"]
combined_condition_txt = features["combined_condition_txt"]
combined_condition_imgtxt = features["combined_condition_imgtxt"]

img_features = torch.from_numpy(img_features)
txt_features = torch.from_numpy(txt_features)
combined_features_origin = torch.from_numpy(combined_features_origin)
combined_condition_img = torch.from_numpy(combined_condition_img)
combined_condition_txt = torch.from_numpy(combined_condition_txt)
combined_condition_imgtxt = torch.from_numpy(combined_condition_imgtxt)

In [21]:
# Compute one-to-one cosine similarity between all four combined features and text features
def element_wise_cosine_similarity(x, y):
    x_norm = F.normalize(x, p=2, dim=-1)
    y_norm = F.normalize(y, p=2, dim=-1)
    cos_sim = (x_norm * y_norm).sum(dim=-1)

    return cos_sim

In [22]:
cos_clip = element_wise_cosine_similarity(img_features, txt_features).mean()
cos_origin = element_wise_cosine_similarity(
    combined_features_origin, txt_features
).mean()
cos_img = element_wise_cosine_similarity(combined_condition_img, txt_features).mean()
cos_txt = element_wise_cosine_similarity(combined_condition_txt, txt_features).mean()
cos_imgtxt = element_wise_cosine_similarity(
    combined_condition_imgtxt, txt_features
).mean()

print(cos_clip, cos_origin, cos_img, cos_txt, cos_imgtxt)

tensor(-0.0037) tensor(-0.0983) tensor(0.3775) tensor(0.4683) tensor(0.4453)


In [27]:
sim_origin = element_wise_cosine_similarity(
    combined_features_origin, img_features
).mean()
sim_img = element_wise_cosine_similarity(combined_condition_img, img_features).mean()
sim_txt = element_wise_cosine_similarity(combined_condition_txt, img_features).mean()
sim_imgtxt = element_wise_cosine_similarity(
    combined_condition_imgtxt, img_features
).mean()

print(sim_origin, sim_img, sim_txt, sim_imgtxt)

tensor(0.0016) tensor(-0.0043) tensor(0.0017) tensor(-0.0027)


In [30]:
cos_clip = F.cosine_similarity(img_features, txt_features, dim=-1)
print(cos_clip.mean())

tensor(-0.0037)


In [31]:
cos_origin = F.cosine_similarity(img_features, combined_features_origin, dim=-1)
print(cos_origin.mean())

tensor(0.0016)
